# Gold Layer: Star Schema
Builds the star schema with fact_order_items and dimension tables (dim_customers, dim_products, dim_sellers).

## 1. Spark Initialization and Configuration

In [12]:
import sys
from pathlib import Path

# Ensure working directory is in path for module imports
work_dir = Path("/home/jovyan/work")
if str(work_dir) not in sys.path:
    sys.path.insert(0, str(work_dir))

from pyspark.sql import SparkSession, Window
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os

# Initialize Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-gold") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


## 2. Load Silver Layer Tables

In [13]:
# Paths
SILVER_DIR = Path("/home/jovyan/work/output/silver")
GOLD_DIR = Path("/home/jovyan/work/output/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

# Read silver tables
orders = spark.read.csv(f"{SILVER_DIR}/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{SILVER_DIR}/order_items.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{SILVER_DIR}/customers.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{SILVER_DIR}/products.csv", header=True, inferSchema=True)
sellers = spark.read.csv(f"{SILVER_DIR}/sellers.csv", header=True, inferSchema=True)
payments = spark.read.csv(f"{SILVER_DIR}/payments.csv", header=True, inferSchema=True)

print("Silver tables loaded")
print(f"  orders: {orders.count()} rows")
print(f"  order_items: {order_items.count()} rows")
print(f"  customers: {customers.count()} rows")
print(f"  products: {products.count()} rows")
print(f"  sellers: {sellers.count()} rows")
print(f"  payments: {payments.count()} rows")

Silver tables loaded
  orders: 1000 rows
  order_items: 1000 rows
  customers: 1000 rows
  products: 1000 rows
  sellers: 1000 rows
  payments: 1000 rows


## 4. Build Dimension Tables

In [14]:
# DIM_CUSTOMERS: One row per customer with location and metrics
# Start with customers (don't want to lose any)
dim_customers = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix"
).distinct()

# Get most recent order's location per customer_unique_id (for those with orders)
# This is optional enrichment; we already have location from customers table
latest_order = orders.join(
    dim_customers,
    on="customer_id",
    how="inner"
).select(
    "customer_unique_id",
    "order_purchase_timestamp",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix"
)

# Get the recent-most location per customer (if they have orders)
window_spec = Window.partitionBy("customer_unique_id").orderBy(desc("order_purchase_timestamp"))
latest_location = latest_order \
    .withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn", "order_purchase_timestamp", "customer_city", "customer_state", "customer_zip_code_prefix")

# Compute customer metrics from orders and order_items (may be sparse)
# Only count what we actually have
customer_metrics = orders.join(
    order_items,
    on="order_id",
    how="inner"
).groupBy("customer_id").agg(
    countDistinct(col("order_id")).alias("total_orders"),
    (sum(col("price")) + sum(col("freight_value"))).alias("total_spend")
)

# Compute first order date from orders alone
first_order = orders.groupBy("customer_id").agg(
    min(col("order_purchase_timestamp")).alias("first_order_date")
)

# Combine metrics
all_metrics = first_order.join(
    customer_metrics,
    on="customer_id",
    how="left"
).fillna(0, subset=["total_orders", "total_spend"])

# Join with customer info and metrics
dim_customers = dim_customers.join(
    all_metrics,
    on="customer_id",
    how="left"
).withColumn(
    "is_repeat_customer",
    when(col("total_orders") > 1, True).otherwise(False)
).withColumn(
    "customer_key",
    row_number().over(Window.orderBy("customer_unique_id"))
).select(
    "customer_key",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    col("first_order_date").cast(TimestampType()),
    col("total_orders").cast(IntegerType()),
    col("total_spend").cast(DecimalType(10, 2)),
    "is_repeat_customer"
)

# Save dim_customers
dim_customers.coalesce(1).write.csv(f"{GOLD_DIR}/dim_customers.csv", header=True, mode="overwrite")
print(f" dim_customers: {dim_customers.count()} rows")

 dim_customers: 1000 rows


In [15]:
# DIM_PRODUCTS: One row per product with derived calculations
dim_products = products.withColumn(
    "product_volume_cm3",
    when(
        col("product_length_cm").isNotNull() & 
        col("product_height_cm").isNotNull() & 
        col("product_width_cm").isNotNull(),
        col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
    ).otherwise(None)
).withColumn(
    "product_key",
    row_number().over(Window.orderBy("product_id"))
).select(
    "product_key",
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_volume_cm3",
    "product_photos_qty",
    "product_description_lenght"
)

# Save dim_products
dim_products.coalesce(1).write.csv(f"{GOLD_DIR}/dim_products.csv", header=True, mode="overwrite")
print(f" dim_products: {dim_products.count()} rows")

 dim_products: 1000 rows


In [16]:
# DIM_SELLERS: One row per seller
dim_sellers = sellers.withColumn(
    "seller_key",
    row_number().over(Window.orderBy("seller_id"))
).select(
    "seller_key",
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
)

# Save dim_sellers
dim_sellers.coalesce(1).write.csv(f"{GOLD_DIR}/dim_sellers.csv", header=True, mode="overwrite")
print(f" dim_sellers: {dim_sellers.count()} rows")

 dim_sellers: 1000 rows


## 5. Calculate Payment Aggregates

In [17]:
# Calculate total order values for payment distribution
order_totals = order_items.groupBy("order_id").agg(
    sum(col("price")).alias("order_total_price")
)

# Join order_items with order totals for proportional distribution
order_items_with_proportions = order_items.join(
    order_totals,
    on="order_id",
    how="inner"
).withColumn(
    "price_proportion",
    col("price") / col("order_total_price")
)

# Aggregate payments at order level
# For each order, get total payment, payment type (highest value, alphabetically tied), and max installments
payment_aggregates = payments.groupBy("order_id").agg(
    sum(col("payment_value")).alias("order_payment_value"),
    max(col("payment_installments")).alias("payment_installments_max")
)

# Get payment type with highest value per order (alphabetically for ties)
payment_type_window = Window.partitionBy("order_id").orderBy(desc("payment_value"), "payment_type")
payment_type_per_order = payments.withColumn(
    "rn", row_number().over(payment_type_window)
).filter(
    col("rn") == 1
).select("order_id", col("payment_type").alias("dominant_payment_type"))

print("Payment aggregates calculated")

Payment aggregates calculated


## 6. Build Fact Table with Star Schema

In [18]:
# Build fact table using LEFT joins to preserve all order_items
# even if they don't match orders/customers (data completeness priority)

# Start with all order_items as the base
fact_order_items = order_items.select(
    "order_id", "order_item_id", "product_id", "seller_id",
    col("price").cast(DecimalType(10, 2)).alias("price"),
    col("freight_value").cast(DecimalType(10, 2)).alias("freight_value")
)

# LEFT join with orders for order-level data
fact_order_items = fact_order_items.join(
    orders.select("order_id", "customer_id", "order_status", 
                  col("order_purchase_timestamp").cast(DateType()).alias("order_date"),
                  "order_delivered_customer_date", "order_estimated_delivery_date"),
    on="order_id",
    how="left"
)

# LEFT join with customers for customer_unique_id and dimension key
fact_order_items = fact_order_items.join(
    customers.select("customer_id", "customer_unique_id"),
    on="customer_id",
    how="left"
)

# LEFT join with dimension tables for surrogate keys
fact_order_items = fact_order_items.join(
    dim_products.select("product_key", "product_id"),
    on="product_id",
    how="left"
).join(
    dim_sellers.select("seller_key", "seller_id"),
    on="seller_id",
    how="left"
).join(
    dim_customers.select("customer_key", "customer_unique_id"),
    on="customer_unique_id",
    how="left"
)

# Add payment info
payment_agg = payments.groupBy("order_id").agg(
    sum(col("payment_value")).alias("order_payment_value"),
    max(col("payment_installments")).alias("payment_installments_max")
)
payment_type_window = Window.partitionBy("order_id").orderBy(desc("payment_value"), "payment_type")
payment_type = payments.withColumn(
    "rn", row_number().over(payment_type_window)
).filter(col("rn") == 1).select("order_id", col("payment_type").alias("payment_type")).drop("rn")

fact_order_items = fact_order_items.join(payment_agg, on="order_id", how="left") \
    .join(payment_type, on="order_id", how="left")

# Calculate proportional payment value
order_totals = order_items.groupBy("order_id").agg(
    sum(col("price")).alias("order_total_price")
)
fact_order_items = fact_order_items.join(order_totals, on="order_id", how="left") \
    .withColumn(
        "payment_value_calc",
        when((col("price").isNotNull() & col("order_total_price").isNotNull() & col("order_payment_value").isNotNull()),
            ((col("price") / col("order_total_price")) * col("order_payment_value")).cast(DecimalType(10, 2))
        ).otherwise(None)
    )

# Calculate delivery metrics
fact_order_items = fact_order_items \
    .withColumn(
        "days_to_deliver",
        when(col("order_delivered_customer_date").isNotNull(),
            datediff(col("order_delivered_customer_date"), col("order_date")).cast(IntegerType())
        ).otherwise(None)
    ) \
    .withColumn(
        "days_delivery_vs_estimate",
        when(col("order_delivered_customer_date").isNotNull(),
            datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")).cast(IntegerType())
        ).otherwise(None)
    ) \
    .withColumn(
        "is_late_delivery",
        when(col("days_delivery_vs_estimate").isNotNull(),
            col("days_delivery_vs_estimate") > 0
        ).otherwise(None)
    )

# Create surrogate key
fact_order_items = fact_order_items.withColumn(
    "order_item_sk",
    row_number().over(Window.orderBy("order_id", "order_item_id"))
)

# Select final columns in order
fact_order_items = fact_order_items.select(
    "order_item_sk",
    "order_id",
    "order_item_id",
    col("customer_key").cast(IntegerType()),
    col("product_key").cast(IntegerType()),
    col("seller_key").cast(IntegerType()),
    "order_date",
    "order_status",
    col("price"),
    col("freight_value"),
    col("payment_value_calc").alias("payment_value"),
    "payment_type",
    col("payment_installments_max").cast(IntegerType()),
    "days_to_deliver",
    "days_delivery_vs_estimate",
    "is_late_delivery"
)

# Save fact table
fact_order_items.coalesce(1).write.csv(f"{GOLD_DIR}/fact_order_items.csv", header=True, mode="overwrite")
print(f" fact_order_items: {fact_order_items.count()} rows")

 fact_order_items: 1000 rows


## 7. Data Quality and Row Count Verification
## & Export Gold Layer Tables to CSV

In [19]:
# Verify gold layer
print("\n" + "="*70)
print("GOLD LAYER VERIFICATION")
print(""*70)

print(f"fact_order_items: {fact_order_items.count()} rows")
print(f"  - Unique orders: {fact_order_items.select('order_id').distinct().count()}")
print(f"  - Unique customers: {fact_order_items.select('customer_key').distinct().count()}")
print(f"  - Unique products: {fact_order_items.select('product_key').distinct().count()}")
print(f"  - Unique sellers: {fact_order_items.select('seller_key').distinct().count()}")

print(f"\ndim_customers: {dim_customers.count()} rows")
print(f"dim_products: {dim_products.count()} rows")
print(f"dim_sellers: {dim_sellers.count()} rows")

print(f"\nGold layer files:")
for file in sorted(Path(GOLD_DIR).glob("*.csv")):
    print(f"  - {file.name}")


GOLD LAYER VERIFICATION

fact_order_items: 1000 rows
  - Unique orders: 893
  - Unique customers: 1
  - Unique products: 21
  - Unique sellers: 160

dim_customers: 1000 rows
dim_products: 1000 rows
dim_sellers: 1000 rows

Gold layer files:
  - dim_customers.csv
  - dim_products.csv
  - dim_sellers.csv
  - fact_order_items.csv


## 8. Star Schema Table Summary

In [20]:
# Sample data verification
print("\nFACT TABLE SAMPLE:")
fact_order_items.limit(5).show()

print("\nDIMENSION CUSTOMER SAMPLE:")
dim_customers.limit(5).show()

print("\nDIMENSION PRODUCT SAMPLE:")
dim_products.limit(5).show()


FACT TABLE SAMPLE:
+-------------+--------------------+-------------+------------+-----------+----------+----------+------------+------+-------------+-------------+------------+------------------------+---------------+-------------------------+----------------+
|order_item_sk|            order_id|order_item_id|customer_key|product_key|seller_key|order_date|order_status| price|freight_value|payment_value|payment_type|payment_installments_max|days_to_deliver|days_delivery_vs_estimate|is_late_delivery|
+-------------+--------------------+-------------+------------+-----------+----------+----------+------------+------+-------------+-------------+------------+------------------------+---------------+-------------------------+----------------+
|            1|00024acbcdf0a6daa...|            1|        NULL|       NULL|      NULL|      NULL|        NULL| 12.99|        12.79|         NULL|        NULL|                    NULL|           NULL|                     NULL|            NULL|
|       

## 9. Final Verification: File Integrity Check

In [21]:
# Verify files are saved
import os
print("\n" + "="*70)
print("FILES SAVED TO GOLD OUTPUT")
print(""*70)
for file in sorted(Path(GOLD_DIR).glob("*.csv")):
    size = os.path.getsize(file) / 1024  # Convert to KB
    print(f" {file.name:30} ({size:.1f} KB)")


FILES SAVED TO GOLD OUTPUT

 dim_customers.csv              (4.0 KB)
 dim_products.csv               (4.0 KB)
 dim_sellers.csv                (4.0 KB)
 fact_order_items.csv           (4.0 KB)


## 10. Gold Layer Pipeline Complete